In [ ]:
# Run parameters
GUIDANCE_SCALE = 3.5
SEED_START = 51        # start seed (seeds 1-50 already exist)
SEED_END = 100         # end seed (inclusive)
N_SEEDS = SEED_END - SEED_START + 1   # = 50 new seeds

In [ ]:
# ============================================================
# KAGGLE NOTEBOOK – Windowed Minority Guidance: FID Sweep
# 250 runs: 5 conditions × 50 seeds (seeds 51-100)
# Generates new samples + computes approximate FID (50 samples/condition)
#
# SETUP (do once before running):
#   1. Add Data → your dataset containing mc_lsun.pt
#   2. Settings → Accelerator → GPU P100 or T4
#   3. Run All
#
# OUTPUT: /kaggle/working/runs_fid_scale3p5_lsun_bedroom.csv
#         /kaggle/working/fid_results_scale3p5.json
# ============================================================

# ── CELL 1: Install dependencies + clone repo + patch ────────
import subprocess, sys, os

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "blobfile", "scikit-learn", "lpips", "mpi4py"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "--force-reinstall", "--no-deps",
                "torch==2.5.1+cu124",
                "--index-url", "https://download.pytorch.org/whl/cu124"], check=True)
# -- GPU smoke test: fail fast if sm_60 not supported ----------------------
_smoke = subprocess.run(
    [sys.executable, "-c",
     "import torch; cap=torch.cuda.get_device_capability(); "
     "sm='sm_{}{}'.format(*cap); archs=torch.cuda.get_arch_list(); "
     "print('[SMOKE] torch={} device={} cap={}'.format("
     "    torch.__version__, torch.cuda.get_device_name(0), sm)); "
     "assert sm in archs, '[FATAL] {} not in {}'.format(sm, archs)"],
    capture_output=True, text=True
)
print(_smoke.stdout.strip() or "(no stdout)")
if _smoke.returncode != 0:
    raise RuntimeError("[FATAL] CUDA smoke test failed:\n" + _smoke.stderr.strip())
print("[SMOKE] PASS")


REPO = "/kaggle/working/minority-guidance"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/soobin-um/minority-guidance", REPO], check=True)
print("Repo ready.")

# Patch 1: add f_extractor_defaults() to script_util.py
script_util_path = f"{REPO}/guided_diffusion/script_util.py"
with open(script_util_path, "r") as f:
    content = f.read()

if "f_extractor_defaults" not in content:
    with open(script_util_path, "a") as f:
        f.write('''

def f_extractor_defaults(use_fp16=False):
    """Defaults for the 256x256 ImageNet classifier used as feature extractor."""
    return dict(
        image_size=256,
        classifier_use_fp16=use_fp16,
        classifier_width=128,
        classifier_depth=2,
        classifier_attention_resolutions="32,16,8",
        classifier_use_scale_shift_norm=True,
        classifier_resblock_updown=True,
        classifier_pool="attention",
        in_channels=3,
        out_channels=1000
    )
''')
    print("Patched script_util.py: appended f_extractor_defaults.")
else:
    print("script_util.py already has f_extractor_defaults.")

# Patch 2: add f_out=False to EncoderUNetModel.forward in unet.py
unet_path = f"{REPO}/guided_diffusion/unet.py"
with open(unet_path, "r") as f:
    lines = f.readlines()

if not any("f_out" in l for l in lines):
    content = "".join(lines)
    old_seq = (
        "        else:\n"
        "            h = h.type(x.dtype)\n"
        "            return self.out(h)\n"
    )
    new_seq = (
        "        else:\n"
        "            h = h.type(x.dtype)\n"
        "            if f_out:\n"
        "                return h\n"
        "            return self.out(h)\n"
    )
    assert old_seq in content, "unet.py patch target not found — repo may have changed"
    content = content.replace(old_seq, new_seq, 1)

    old_sig = "    def forward(self, x, timesteps):\n"
    new_sig = "    def forward(self, x, timesteps, f_out=False):\n"
    assert content.count(old_sig) == 1, f"Expected exactly 1 forward signature, got {content.count(old_sig)}"
    content = content.replace(old_sig, new_sig, 1)

    with open(unet_path, "w") as f:
        f.write(content)
    print("Patched unet.py: added f_out parameter.")
else:
    print("unet.py already has f_out parameter.")


# ── CELL 2: Write custom scripts ─────────────────────────────
os.makedirs(f"{REPO}/scripts", exist_ok=True)

# windowed_classifier_sample.py
with open(f"{REPO}/windowed_classifier_sample.py", "w") as f:
    f.write('''\
"""
windowed_classifier_sample.py
Extends classifier_sample.py with --t_start / --t_end flags that gate
minority guidance to a contiguous window of the denoising chain.
Guidance applied only when: t_start <= current_timestep < t_end
"""

import argparse
import os
import sys

import numpy as np
import torch as th
import torch.distributed as dist
import torch.nn.functional as F

from guided_diffusion import dist_util, logger
from guided_diffusion.script_util import (
    model_and_diffusion_defaults,
    classifier_defaults,
    create_model_and_diffusion,
    create_classifier,
    add_dict_to_argparser,
    args_to_dict,
    f_extractor_defaults,
)


def _dist_ready():
    return dist.is_available() and dist.is_initialized()

def _world_size():
    return dist.get_world_size() if _dist_ready() else 1

def _rank():
    return dist.get_rank() if _dist_ready() else 0


def main():
    args = create_argparser().parse_args()
    th.set_num_threads(8)

    if args.seed != "":
        th.manual_seed(int(args.seed))
        np.random.seed(int(args.seed))

    dist_util.setup_dist()
    logger.configure()

    logger.log("creating model and diffusion...")
    model, diffusion = create_model_and_diffusion(
        **args_to_dict(args, model_and_diffusion_defaults().keys())
    )
    model.load_state_dict(
        dist_util.load_state_dict(args.model_path, map_location="cpu")
    )
    model.to(dist_util.dev())
    if args.use_fp16:
        model.convert_to_fp16()
    model.eval()

    args.image_size = args.latent_size
    logger.log("loading classifier...")
    classifier = create_classifier(**args_to_dict(args, classifier_defaults().keys()))
    classifier.load_state_dict(
        dist_util.load_state_dict(args.classifier_path, map_location="cpu")
    )
    classifier.to(dist_util.dev())
    if args.classifier_use_fp16:
        classifier.convert_to_fp16()
    classifier.eval()

    args.image_size = f_extractor_defaults()["image_size"]
    logger.log("loading feature extractor...")
    f_extractor = create_classifier(**f_extractor_defaults())
    f_extractor.load_state_dict(
        dist_util.load_state_dict(args.f_extractor_path, map_location="cpu")
    )
    f_extractor.to(dist_util.dev())
    if args.classifier_use_fp16:
        f_extractor.convert_to_fp16()
    f_extractor.eval()

    t_start = args.t_start
    t_end = args.t_end
    logger.log(f"guidance window: t in [{t_start}, {t_end}), scale={args.classifier_scale}")

    def cond_fn(x, t, y=None):
        assert y is not None
        current_t = int(t[0].item())
        if not (t_start <= current_t < t_end):
            return th.zeros_like(x)
        with th.enable_grad():
            x_in = x.detach().requires_grad_(True)
            latents = f_extractor(x_in, timesteps=t, f_out=True)
            logits = classifier(latents, timesteps=t)
            log_probs = F.log_softmax(logits, dim=-1)
            selected = log_probs[range(len(logits)), y.view(-1)]
            return th.autograd.grad(selected.sum(), x_in)[0] * args.classifier_scale

    def model_fn(x, t, y=None):
        assert y is not None
        return model(x, t, y if args.class_cond else None)

    logger.log("sampling...")
    all_images = []
    all_labels = []
    while len(all_images) * args.batch_size < args.num_samples:
        model_kwargs = {}
        if args.use_manual_class:
            classes = args.manual_class_id * th.ones(
                size=(args.batch_size,), device=dist_util.dev()
            ).long()
        else:
            classes = th.randint(
                low=args.rand_starting_class_id,
                high=args.num_classes,
                size=(args.batch_size,),
                device=dist_util.dev(),
            )
        model_kwargs["y"] = classes
        sample_fn = (
            diffusion.p_sample_loop if not args.use_ddim else diffusion.ddim_sample_loop
        )
        sample = sample_fn(
            model_fn,
            (args.batch_size, 3, args.image_size, args.image_size),
            clip_denoised=args.clip_denoised,
            model_kwargs=model_kwargs,
            cond_fn=cond_fn,
            device=dist_util.dev(),
        )
        sample = ((sample + 1) * 127.5).clamp(0, 255).to(th.uint8)
        sample = sample.permute(0, 2, 3, 1)
        sample = sample.contiguous()

        if _dist_ready():
            gathered_samples = [th.zeros_like(sample) for _ in range(_world_size())]
            dist.all_gather(gathered_samples, sample)
            all_images.extend([s.cpu().numpy() for s in gathered_samples])
            gathered_labels = [th.zeros_like(classes) for _ in range(_world_size())]
            dist.all_gather(gathered_labels, classes)
            all_labels.extend([l.cpu().numpy() for l in gathered_labels])
        else:
            all_images.append(sample.cpu().numpy())
            all_labels.append(classes.cpu().numpy())
        logger.log(f"created {len(all_images) * args.batch_size} samples")

    arr = np.concatenate(all_images, axis=0)[: args.num_samples]
    label_arr = np.concatenate(all_labels, axis=0)[: args.num_samples]
    if _rank() == 0:
        shape_str = "x".join([str(x) for x in arr.shape])
        out_path = os.path.join(logger.get_dir(), f"samples_{shape_str}.npz")
        logger.log(f"saving to {out_path}")
        np.savez(out_path, arr, label_arr)

    if _dist_ready():
        dist.barrier()
    logger.log("sampling complete")


def create_argparser():
    defaults = dict(
        clip_denoised=True,
        num_samples=10000,
        batch_size=16,
        use_ddim=False,
        model_path="",
        classifier_path="",
        classifier_scale=1.0,
        seed="",
        num_classes=100,
        use_manual_class=False,
        manual_class_id=0,
        rand_starting_class_id=0,
        f_extractor_path="",
        latent_size=8,
        t_start=0,
        t_end=1000,
    )
    defaults.update(model_and_diffusion_defaults())
    defaults.update(classifier_defaults())
    parser = argparse.ArgumentParser()
    add_dict_to_argparser(parser, defaults)
    return parser


if __name__ == "__main__":
    main()
''')

# scripts/extract_metrics.py
with open(f"{REPO}/scripts/extract_metrics.py", "w") as f:
    f.write('''\
import argparse, json, sys, os
import numpy as np
import torch
import torch.nn.functional as F

SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
REPO_DIR = os.path.dirname(SCRIPT_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from guided_diffusion.script_util import (
    create_classifier, classifier_defaults, f_extractor_defaults,
)
from guided_diffusion import dist_util


def load_classifier(path, cfg):
    model = create_classifier(**cfg)
    state = dist_util.load_state_dict(path, map_location="cpu")
    model.load_state_dict(state)
    model.eval()
    return model


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--npz_path", required=True)
    parser.add_argument("--classifier_path", required=True)
    parser.add_argument("--f_extractor_path", required=True)
    parser.add_argument("--class_id", type=int, default=99)
    args = parser.parse_args()

    data = np.load(args.npz_path)
    images_uint8 = data["arr_0"]
    n = images_uint8.shape[0]

    images = torch.from_numpy(images_uint8).float()
    images = images.permute(0, 3, 1, 2)
    images = images / 127.5 - 1.0

    f_cfg = f_extractor_defaults()
    f_extractor = load_classifier(args.f_extractor_path, f_cfg)

    clf_cfg = {**classifier_defaults(), "image_size": 8, "in_channels": 512, "out_channels": 100}
    classifier = load_classifier(args.classifier_path, clf_cfg)

    device = torch.device("cpu")
    f_extractor.to(device)
    classifier.to(device)

    t_zeros = torch.zeros(n, dtype=torch.long, device=device)
    labels = torch.full((n,), args.class_id, dtype=torch.long, device=device)

    confidences, losses = [], []
    with torch.no_grad():
        for i in range(n):
            img = images[i:i+1].to(device)
            t = t_zeros[i:i+1]
            label = labels[i:i+1]
            latents = f_extractor(img, timesteps=t, f_out=True)
            logits = classifier(latents, timesteps=t)
            probs = F.softmax(logits, dim=-1)
            conf = probs[0, args.class_id].item()
            loss = F.cross_entropy(logits, label).item()
            confidences.append(conf)
            losses.append(loss)

    print(json.dumps({
        "n": n,
        "classifier_mean_confidence": float(np.mean(confidences)),
        "classifier_mean_loss": float(np.mean(losses)),
        "confidences": confidences,
        "losses": losses,
    }))


if __name__ == "__main__":
    main()
''')

# scripts/run_experiment.py
with open(f"{REPO}/scripts/run_experiment.py", "w") as f:
    f.write('''\
import argparse, csv, json, os, subprocess, sys, time
from datetime import datetime, timezone

REPO_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
DATA_CSV = os.path.join(REPO_DIR, "data", "experiment_runs.csv")
RESULTS_DIR = os.path.join(REPO_DIR, "results")
RUNTIME_LOG = os.path.join(REPO_DIR, "logs", "runtime.log")

CSV_FIELDS = [
    "run_id", "condition", "guidance_window", "t_start", "t_end",
    "guidance_scale", "seed", "n_generated",
    "classifier_mean_confidence", "classifier_mean_loss",
    "runtime_seconds", "artifact_path", "timestamp",
]


def log(msg):
    ts = datetime.now(timezone.utc).isoformat()
    line = f"[{ts}] {msg}"
    print(line, flush=True)
    os.makedirs(os.path.dirname(RUNTIME_LOG), exist_ok=True)
    with open(RUNTIME_LOG, "a", encoding="utf-8") as f:
        f.write(line + "\\n")


def load_existing_run_ids():
    if not os.path.exists(DATA_CSV):
        return set()
    with open(DATA_CSV, "r", newline="") as f:
        reader = csv.DictReader(f)
        return {row["run_id"] for row in reader}


def write_csv_row(row):
    os.makedirs(os.path.dirname(DATA_CSV), exist_ok=True)
    write_header = not os.path.exists(DATA_CSV)
    with open(DATA_CSV, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_FIELDS)
        if write_header:
            writer.writeheader()
        writer.writerow(row)


def find_npz(result_dir):
    if not os.path.isdir(result_dir):
        return None
    for fname in os.listdir(result_dir):
        if fname.endswith(".npz"):
            return os.path.join(result_dir, fname)
    return None


def extract_metrics(npz_path):
    script = os.path.join(REPO_DIR, "scripts", "extract_metrics.py")
    result = subprocess.run(
        [sys.executable, script,
         "--npz_path", npz_path,
         "--classifier_path", "ckpt/mc_lsun.pt",
         "--f_extractor_path", "ckpt/256x256_classifier.pt",
         "--class_id", "99"],
        capture_output=True, text=True, cwd=REPO_DIR
    )
    if result.returncode != 0:
        raise RuntimeError(f"extract_metrics failed:\\n{result.stderr}")
    return json.loads(result.stdout.strip())


def run_sampler(run_id, condition, guidance_window, guidance_scale,
                t_start, t_end, seed, num_samples, result_dir):
    os.makedirs(result_dir, exist_ok=True)
    log_path = os.path.join(result_dir, "stdout.log")
    err_path = os.path.join(result_dir, "stderr.log")
    rel_result_dir = os.path.relpath(result_dir, REPO_DIR)

    cmd = [
        sys.executable, "windowed_classifier_sample.py",
        "--attention_resolutions", "32,16,8",
        "--class_cond", "False",
        "--diffusion_steps", "1000",
        "--dropout", "0.1",
        "--image_size", "256",
        "--learn_sigma", "True",
        "--noise_schedule", "linear",
        "--num_channels", "256",
        "--num_head_channels", "64",
        "--num_res_blocks", "2",
        "--resblock_updown", "True",
        "--use_fp16", "False",
        "--use_scale_shift_norm", "True",
        "--latent_size", "8",
        "--in_channels", "512",
        "--out_channels", "100",
        "--classifier_attention_resolutions", "8",
        "--classifier_depth", "2",
        "--classifier_width", "128",
        "--classifier_pool", "attention",
        "--classifier_resblock_updown", "False",
        "--classifier_use_scale_shift_norm", "True",
        "--classifier_use_fp16", "False",
        "--batch_size", "1",
        "--num_samples", str(num_samples),
        "--timestep_respacing", "250",
        "--classifier_scale", str(guidance_scale),
        "--use_manual_class", "True",
        "--manual_class_id", "99",
        "--seed", str(seed),
        "--t_start", str(t_start),
        "--t_end", str(t_end),
        "--model_path", "ckpt/lsun_bedroom.pt",
        "--classifier_path", "ckpt/mc_lsun.pt",
        "--f_extractor_path", "ckpt/256x256_classifier.pt",
    ]

    env = os.environ.copy()
    env["OPENAI_LOGDIR"] = rel_result_dir

    log(f"[{run_id}] window={guidance_window} t=[{t_start},{t_end}) scale={guidance_scale} seed={seed} n={num_samples}")
    t0 = time.time()
    with open(log_path, "w") as out_f, open(err_path, "w") as err_f:
        proc = subprocess.run(cmd, stdout=out_f, stderr=err_f, cwd=REPO_DIR, env=env)
    runtime = time.time() - t0

    if proc.returncode != 0:
        with open(err_path) as ef:
            err_tail = ef.read()[-1500:]
        raise RuntimeError(f"Sampler exit {proc.returncode}:\\n{err_tail}")

    npz = find_npz(result_dir)
    if npz is None:
        raise RuntimeError(f"Sampler completed but no .npz in {result_dir}")

    log(f"[{run_id}] Done in {runtime:.1f}s -> {npz}")
    return npz, runtime


def execute_run(run_spec, existing_ids, dry_run=False):
    run_id = run_spec["run_id"]
    if run_id in existing_ids:
        log(f"[{run_id}] Already in CSV - skipping.")
        return

    condition       = run_spec["condition"]
    guidance_window = run_spec["guidance_window"]
    guidance_scale  = run_spec["guidance_scale"]
    t_start         = run_spec["t_start"]
    t_end           = run_spec["t_end"]
    seed            = run_spec["seed"]
    num_samples     = run_spec.get("num_samples", 1)
    result_dir      = os.path.join(RESULTS_DIR, run_id)

    if dry_run:
        log(f"[DRY-RUN] {run_id}: window={guidance_window} t=[{t_start},{t_end}) seed={seed}")
        return

    pre_npz = find_npz(result_dir)
    if pre_npz:
        log(f"[{run_id}] Pre-existing .npz found - extracting metrics only.")
        npz_path, runtime = pre_npz, None
    else:
        npz_path, runtime = run_sampler(
            run_id, condition, guidance_window, guidance_scale,
            t_start, t_end, seed, num_samples, result_dir
        )

    log(f"[{run_id}] Extracting metrics...")
    metrics = extract_metrics(npz_path)

    row = {
        "run_id": run_id, "condition": condition,
        "guidance_window": guidance_window, "t_start": t_start, "t_end": t_end,
        "guidance_scale": guidance_scale, "seed": seed,
        "n_generated": metrics["n"],
        "classifier_mean_confidence": metrics["classifier_mean_confidence"],
        "classifier_mean_loss": metrics["classifier_mean_loss"],
        "runtime_seconds": runtime if runtime is not None else "",
        "artifact_path": npz_path,
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }

    required = ["run_id", "condition", "guidance_window", "t_start", "t_end",
                "guidance_scale", "seed", "n_generated",
                "classifier_mean_confidence", "classifier_mean_loss"]
    for field in required:
        if row[field] == "" or row[field] is None:
            raise ValueError(f"Incomplete row for {run_id}: field \'{field}\' missing.")

    write_csv_row(row)
    existing_ids.add(run_id)
    log(f"[{run_id}] CSV OK  confidence={row[\'classifier_mean_confidence\']:.4f}  loss={row[\'classifier_mean_loss\']:.4f}")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--dry-run", action="store_true")
    parser.add_argument("--matrix", required=True)
    args = parser.parse_args()

    with open(args.matrix) as f:
        matrix = json.load(f)

    existing_ids = load_existing_run_ids()
    log(f"Existing CSV run_ids: {sorted(existing_ids) or \'(none)\'}")  

    runs = sorted(matrix["planned_runs"], key=lambda r: r.get("priority", 99))
    for run_spec in runs:
        try:
            execute_run(run_spec, existing_ids, dry_run=args.dry_run)
        except Exception as e:
            log(f"[{run_spec[\'run_id\']}] ERROR: {e}")
            log("Halting.")
            sys.exit(1)

    log("All targeted runs complete.")


if __name__ == "__main__":
    main()
''')

print("Custom scripts written.")


# ── CELL 3: Locate and copy checkpoints ──────────────────────
import shutil, glob

ckpt_dir = f"{REPO}/ckpt"
os.makedirs(ckpt_dir, exist_ok=True)

# mc_lsun.pt – recursive search: works regardless of Kaggle dataset mount depth
candidates = glob.glob("/kaggle/input/**/mc_lsun.pt", recursive=True)
if candidates:
    shutil.copy(candidates[0], f"{ckpt_dir}/mc_lsun.pt")
    print(f"mc_lsun.pt copied from: {candidates[0]}")
else:
    # Diagnostic: print everything under /kaggle/input/ so you can see what's there
    all_files = []
    for root, dirs, files in os.walk("/kaggle/input"):
        for fn in files:
            all_files.append(os.path.join(root, fn))
    print("=== ALL FILES UNDER /kaggle/input/ ===")
    for p in all_files:
        print(p)
    raise FileNotFoundError(
        "mc_lsun.pt not found anywhere under /kaggle/input/.\n"
        "Make sure you added the dataset containing mc_lsun.pt to this notebook."
    )

# lsun_bedroom.pt (~2.1 GB) from OpenAI public storage
if not os.path.exists(f"{ckpt_dir}/lsun_bedroom.pt"):
    print("Downloading lsun_bedroom.pt (~2.1 GB)...")
    subprocess.run([
        "wget", "-q", "--show-progress",
        "-O", f"{ckpt_dir}/lsun_bedroom.pt",
        "https://openaipublic.blob.core.windows.net/diffusion/jul-2021/lsun_bedroom.pt"
    ], check=True)
    print("lsun_bedroom.pt done.")
else:
    print("lsun_bedroom.pt already present.")

# 256x256_classifier.pt (~206 MB) from OpenAI public storage
if not os.path.exists(f"{ckpt_dir}/256x256_classifier.pt"):
    print("Downloading 256x256_classifier.pt (~206 MB)...")
    subprocess.run([
        "wget", "-q", "--show-progress",
        "-O", f"{ckpt_dir}/256x256_classifier.pt",
        "https://openaipublic.blob.core.windows.net/diffusion/jul-2021/256x256_classifier.pt"
    ], check=True)
    print("256x256_classifier.pt done.")
else:
    print("256x256_classifier.pt already present.")

print("All checkpoints ready.")


# ── CELL 4: Build matrix and run 250 experiments (seeds 51-100) ─
import json

SEEDS = list(range(SEED_START, SEED_END + 1))  # 51..100
NUM_SAMPLES = 1

CONDITIONS = [
    {"condition": "baseline",       "guidance_window": "none",  "guidance_scale": 0.0,            "t_start": 0,   "t_end": 0},
    {"condition": "minority-full",  "guidance_window": "full",  "guidance_scale": GUIDANCE_SCALE, "t_start": 0,   "t_end": 1000},
    {"condition": "minority-early", "guidance_window": "early", "guidance_scale": GUIDANCE_SCALE, "t_start": 0,   "t_end": 333},
    {"condition": "minority-mid",   "guidance_window": "mid",   "guidance_scale": GUIDANCE_SCALE, "t_start": 333, "t_end": 667},
    {"condition": "minority-late",  "guidance_window": "late",  "guidance_scale": GUIDANCE_SCALE, "t_start": 667, "t_end": 1000},
]

COND_SHORT = {
    "baseline": "baseline",
    "minority-full": "full",
    "minority-early": "early",
    "minority-mid": "mid",
    "minority-late": "late",
}

planned_runs = []
priority = 1
for seed in SEEDS:
    for cond in CONDITIONS:
        short = COND_SHORT[cond["condition"]]
        planned_runs.append({
            "run_id": f"{short}_seed{seed}",
            "condition": cond["condition"],
            "guidance_window": cond["guidance_window"],
            "guidance_scale": cond["guidance_scale"],
            "t_start": cond["t_start"],
            "t_end": cond["t_end"],
            "seed": seed,
            "num_samples": NUM_SAMPLES,
            "priority": priority,
        })
        priority += 1

matrix = {"planned_runs": planned_runs}
matrix_path = f"{REPO}/manifests/kaggle_matrix.json"
os.makedirs(os.path.dirname(matrix_path), exist_ok=True)
with open(matrix_path, "w") as f:
    json.dump(matrix, f, indent=2)

print(f"Matrix: {len(planned_runs)} runs – seeds {SEEDS[0]}..{SEEDS[-1]}, {len(CONDITIONS)} conditions")
print("Starting experiment...\n")

result = subprocess.run(
    [sys.executable, "scripts/run_experiment.py", "--matrix", matrix_path],
    cwd=REPO
)

if result.returncode != 0:
    print("\nERROR: orchestrator exited non-zero. Check logs above.")
else:
    print("\nAll runs complete!")


# ── CELL 5: Save and display results ─────────────────────────
import pandas as pd

csv_src = f"{REPO}/data/experiment_runs.csv"
csv_dst = "/kaggle/working/runs_fid_scale3p5_lsun_bedroom.csv"

if os.path.exists(csv_src):
    shutil.copy(csv_src, csv_dst)
    df = pd.read_csv(csv_src)
    print(f"\n{len(df)} rows written to CSV.\n")
    print(df[["run_id", "condition", "guidance_window", "seed",
              "classifier_mean_confidence", "classifier_mean_loss",
              "runtime_seconds"]].to_string(index=False))
    print(f"\nDownload from: {csv_dst}")
else:
    print("CSV not found – check for errors above.")

In [ ]:
# ── FID COMPUTATION ─────────────────────────────────────────────────────
# Computes approximate FID using 50 samples per condition (seeds 51-100).
# NOTE: FID with 50 samples is a rough estimate; standard practice uses >=2k.
# Combined FID with seeds 1-50 requires those .npz files, which are not
# available from the CSV dataset alone — re-run both batches if needed.
#
# Reference: LSUN Bedroom validation set (~300 images, ~135MB)
# Requires: pytorch-fid, lmdb, Pillow

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytorch-fid", "lmdb"], check=True)

import numpy as np
import os
import shutil
import urllib.request
from pathlib import Path
from PIL import Image
import torch
from pytorch_fid import fid_score

WORK = "/kaggle/working"
REF_DIR = f"{WORK}/fid_reference"
os.makedirs(REF_DIR, exist_ok=True)

# --- Download LSUN Bedroom val set (lmdb ~135MB) ---
LSUN_URL = "http://dl.yf.io/lsun/scenes/bedroom_val_lmdb.zip"
lmdb_zip = f"{WORK}/bedroom_val_lmdb.zip"
if not os.path.exists(lmdb_zip):
    print("Downloading LSUN Bedroom val set...")
    urllib.request.urlretrieve(LSUN_URL, lmdb_zip)
    print("Downloaded.")

import zipfile
lmdb_dir = f"{WORK}/bedroom_val_lmdb"
if not os.path.exists(lmdb_dir):
    with zipfile.ZipFile(lmdb_zip, 'r') as z:
        z.extractall(WORK)
    print("Extracted LMDB.")

# Extract images from LMDB
import lmdb
import io
env = lmdb.open(lmdb_dir, readonly=True, lock=False)
n_extracted = 0
MAX_REF = 300  # use all val images
with env.begin() as txn:
    cursor = txn.cursor()
    for key, val in cursor:
        img = Image.open(io.BytesIO(val)).convert("RGB").resize((256, 256), Image.LANCZOS)
        img.save(f"{REF_DIR}/{n_extracted:04d}.png")
        n_extracted += 1
        if n_extracted >= MAX_REF:
            break
print(f"Extracted {n_extracted} reference images to {REF_DIR}")

# --- Collect generated images for each condition (seeds 51-100 from this run) ---
import glob as glob_mod

def npz_to_pngs(npz_path, out_dir, prefix=""):
    os.makedirs(out_dir, exist_ok=True)
    data = np.load(npz_path)
    imgs = data["arr_0"]  # uint8, shape (N, H, W, 3)
    paths = []
    for i, img in enumerate(imgs):
        p = f"{out_dir}/{prefix}{i:04d}.png"
        Image.fromarray(img).save(p)
        paths.append(p)
    return paths

COND_DIRS = {}
for cond in ["baseline", "minority-full", "minority-early", "minority-mid", "minority-late"]:
    short = {"baseline": "baseline", "minority-full": "full", "minority-early": "early",
             "minority-mid": "mid", "minority-late": "late"}[cond]
    out_dir = f"{WORK}/fid_gen_{short}"
    os.makedirs(out_dir, exist_ok=True)
    # Seeds 51-100 from this run
    for seed in range(SEED_START, SEED_END + 1):
        run_id = f"{short}_seed{seed}"
        npzs = glob_mod.glob(f"{REPO}/results/{run_id}/*.npz")
        if npzs:
            npz_to_pngs(npzs[0], out_dir, prefix=f"s{seed}_")

    n = len(os.listdir(out_dir))
    print(f"  {cond}: {n} images in {out_dir}")
    COND_DIRS[cond] = out_dir

# --- Compute FID for each condition vs. reference ---
print("\nComputing FID scores (approx — 50 samples/condition)...")
fid_results = {}
for cond, gen_dir in COND_DIRS.items():
    n_gen = len(os.listdir(gen_dir))
    n_ref = len(os.listdir(REF_DIR))
    if n_gen < 10:
        print(f"  {cond}: only {n_gen} images, skipping FID")
        fid_results[cond] = None
        continue
    try:
        fid_val = fid_score.calculate_fid_given_paths(
            [gen_dir, REF_DIR],
            batch_size=50,
            device="cuda" if torch.cuda.is_available() else "cpu",
            dims=2048,
        )
        fid_results[cond] = round(fid_val, 2)
        print(f"  {cond}: FID = {fid_val:.2f} (n_gen={n_gen}, n_ref={n_ref})")
    except Exception as e:
        print(f"  {cond}: FID failed: {e}")
        fid_results[cond] = None

# Save FID results
import json
fid_out = f"{WORK}/fid_results_scale3p5.json"
with open(fid_out, "w") as f:
    json.dump({
        "scale": GUIDANCE_SCALE,
        "n_seeds_new": N_SEEDS,
        "seed_range": f"{SEED_START}-{SEED_END}",
        "n_ref_images": n_extracted,
        "note": "FID computed on 50 samples/condition (seeds 51-100 only). Approximate.",
        "fid_scores": fid_results,
    }, f, indent=2)
print(f"\nFID results saved to {fid_out}")
print(json.dumps(fid_results, indent=2))